# MV Performance Analysis by Employee, Account, and Employee-Account

This notebook calculates Recall and FPR at three levels:

1. Employee (`login_id`)
2. Account (`acct_nbr`)
3. Employee-account (`login_id`, `acct_nbr`)

For each level, it first calculates Recall and FPR for every group, then calculates:

- simple average
- weighted average

The fixed threshold is **0.54**.

The notebook only loads the saved LightGBM model and saved feature tables. It does not retrain the model, rerun Optuna, or reselect the threshold.


In [ ]:
import numpy as np
import pandas as pd
import pyspark.sql.functions as F

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## Load project utility functions

This only loads helper functions. It does not save or retrain anything.


In [ ]:
%run ../utils/load_functions_utils

## Configuration

In [ ]:
BEST_THRESHOLD = 0.54
TARGET = "label"

IN_TIME_FEATURE_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_features_table_v2"
)

OOT_FEATURE_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_features_table_6m_v2"
)

MODEL_ARTIFACT_NAME = "lgbm_model.pkl"
MODEL_ARTIFACT_VERSION = 1
MODEL_ARTIFACT_TYPE = "model"
MODEL_TMP_PATH = "tmp/insider_us_nms"

FEATURES = [
    "inquiries",
    "maintenances",
    "prop_maintenances",
    "prop_inquiries",
    "after_hours_touches",
    "emp_prop_acct_seqs",
    "avg_time_bw_touch_dt",
    "mean_acct_balance_touch",
    "change_acct_balance",
    "num_status_changes",
    "mean_seq_len_in_time",
    "stddev_acct_balance",
    "avg_emp_touches_per_acct",
    "mean_acct_balance_touch_deviation",
    "emp_prop_touch_seq_ratio",
    "stddev_acct_balance_emp",
    "acct_prop_touch_seq_ratio",
    "num_contact_info_changes",
    "avg_emp_behavior_deviation",
    "num_address_changes"
]

## Load saved test and OOT data

The in-time table contains train, validation, and test rows. Only the original test split is used here.


In [ ]:
features_spk = spark.read.format("delta").load(IN_TIME_FEATURE_PATH)
oot_spk = spark.read.format("delta").load(OOT_FEATURE_PATH)

test_spk = features_spk.filter(F.col("label_split") == "test")

required_columns = ["login_id", "acct_nbr", TARGET] + FEATURES

for dataset_name, df in [
    ("In-time test", test_spk),
    ("Out-of-time", oot_spk)
]:
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{dataset_name} is missing columns: {missing}")

test_pd = test_spk.select(*required_columns).toPandas()
oot_pd = oot_spk.select(*required_columns).toPandas()

print("In-time test shape:", test_pd.shape)
print("Out-of-time shape:", oot_pd.shape)

## Load the saved LightGBM model

This is inference only. No model fitting is performed.


In [ ]:
final_model = load_artifact_as_file(
    MODEL_ARTIFACT_NAME,
    MODEL_ARTIFACT_VERSION,
    MODEL_ARTIFACT_TYPE,
    tmp_file_path=MODEL_TMP_PATH
)

print("Loaded model type:", type(final_model))

if hasattr(final_model, "feature_name_"):
    saved_feature_order = list(final_model.feature_name_)
    if saved_feature_order != FEATURES:
        raise ValueError(
            "Feature order does not match the saved LightGBM model."
        )

## Generate probabilities and predictions

The fixed threshold is 0.54:

- probability >= 0.54 -> prediction = 1
- probability < 0.54 -> prediction = 0


In [ ]:
for df in [test_pd, oot_pd]:
    df["probability"] = final_model.predict_proba(df[FEATURES])[:, 1]
    df["prediction"] = (
        df["probability"] >= BEST_THRESHOLD
    ).astype(int)

display(test_pd.head(10))

## Function to calculate Recall and FPR for each group

For each employee, account, or employee-account pair:

- TP = actual 1 and predicted 1
- FN = actual 1 and predicted 0
- FP = actual 0 and predicted 1
- TN = actual 0 and predicted 0

Then:

```text
Recall = TP / (TP + FN)
FPR    = FP / (FP + TN)
```

If a group has no positive rows, Recall is undefined and returned as `NaN`.

If a group has no negative rows, FPR is undefined and returned as `NaN`.


In [ ]:
def calculate_group_metrics(df, group_cols):
    results = []

    for group_key, group_df in df.groupby(group_cols, dropna=False):
        tp = (
            (group_df[TARGET] == 1)
            & (group_df["prediction"] == 1)
        ).sum()

        fn = (
            (group_df[TARGET] == 1)
            & (group_df["prediction"] == 0)
        ).sum()

        fp = (
            (group_df[TARGET] == 0)
            & (group_df["prediction"] == 1)
        ).sum()

        tn = (
            (group_df[TARGET] == 0)
            & (group_df["prediction"] == 0)
        ).sum()

        positive_count = tp + fn
        negative_count = fp + tn

        recall = tp / positive_count if positive_count > 0 else np.nan
        fpr = fp / negative_count if negative_count > 0 else np.nan

        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        result = {
            group_cols[i]: group_key[i]
            for i in range(len(group_cols))
        }

        result.update({
            "row_count": len(group_df),
            "positive_count": int(positive_count),
            "negative_count": int(negative_count),
            "tp": int(tp),
            "fn": int(fn),
            "fp": int(fp),
            "tn": int(tn),
            "recall": recall,
            "fpr": fpr
        })

        results.append(result)

    return pd.DataFrame(results)

## Functions for simple and weighted averages

Simple average gives equal weight to every group.

Weighted average Recall uses the number of actual positive rows as the weight.

Weighted average FPR uses the number of actual negative rows as the weight.


In [ ]:
def weighted_average(values, weights):
    valid = values.notna() & weights.notna() & (weights > 0)

    if valid.sum() == 0:
        return np.nan

    return np.average(
        values[valid],
        weights=weights[valid]
    )


def summarize_group_metrics(group_metrics, dataset_name, analysis_level):
    return pd.DataFrame([{
        "dataset": dataset_name,
        "analysis_level": analysis_level,
        "number_of_groups": len(group_metrics),
        "groups_with_defined_recall": group_metrics["recall"].notna().sum(),
        "groups_with_defined_fpr": group_metrics["fpr"].notna().sum(),
        "simple_average_recall": group_metrics["recall"].mean(),
        "weighted_average_recall": weighted_average(
            group_metrics["recall"],
            group_metrics["positive_count"]
        ),
        "simple_average_fpr": group_metrics["fpr"].mean(),
        "weighted_average_fpr": weighted_average(
            group_metrics["fpr"],
            group_metrics["negative_count"]
        )
    }])

# A. Employee-level performance

Grouping key: `login_id`


In [ ]:
test_employee_metrics = calculate_group_metrics(
    test_pd,
    ["login_id"]
)

oot_employee_metrics = calculate_group_metrics(
    oot_pd,
    ["login_id"]
)

test_employee_summary = summarize_group_metrics(
    test_employee_metrics,
    "In-Time Test",
    "Employee"
)

oot_employee_summary = summarize_group_metrics(
    oot_employee_metrics,
    "Out-of-Time",
    "Employee"
)

display(test_employee_metrics)
display(oot_employee_metrics)

# B. Account-level performance

Grouping key: `acct_nbr`


In [ ]:
test_account_metrics = calculate_group_metrics(
    test_pd,
    ["acct_nbr"]
)

oot_account_metrics = calculate_group_metrics(
    oot_pd,
    ["acct_nbr"]
)

test_account_summary = summarize_group_metrics(
    test_account_metrics,
    "In-Time Test",
    "Account"
)

oot_account_summary = summarize_group_metrics(
    oot_account_metrics,
    "Out-of-Time",
    "Account"
)

display(test_account_metrics)
display(oot_account_metrics)

# C. Employee-account-level performance

Grouping keys: `login_id + acct_nbr`


In [ ]:
test_pair_metrics = calculate_group_metrics(
    test_pd,
    ["login_id", "acct_nbr"]
)

oot_pair_metrics = calculate_group_metrics(
    oot_pd,
    ["login_id", "acct_nbr"]
)

test_pair_summary = summarize_group_metrics(
    test_pair_metrics,
    "In-Time Test",
    "Employee-Account"
)

oot_pair_summary = summarize_group_metrics(
    oot_pair_metrics,
    "Out-of-Time",
    "Employee-Account"
)

display(test_pair_metrics)
display(oot_pair_metrics)

## Final summary

This is the main output for the validation request.


In [ ]:
performance_summary = pd.concat(
    [
        test_employee_summary,
        oot_employee_summary,
        test_account_summary,
        oot_account_summary,
        test_pair_summary,
        oot_pair_summary
    ],
    ignore_index=True
)

display(performance_summary)

## Interpretation

- `simple_average_recall`: every group receives equal weight
- `weighted_average_recall`: groups with more positive rows receive more weight
- `simple_average_fpr`: every group receives equal weight
- `weighted_average_fpr`: groups with more negative rows receive more weight

Groups with no positive rows are excluded from Recall averages.

Groups with no negative rows are excluded from FPR averages.
